# Carga de datos del CSV

## Introducción ##

El siguiente documento detalla el análisis de datos de eBird.

### ¿Cómo usar este documento? ###

Este documento hace referencia al proceso de generación de un entorno de análisis de datos, de acuerdo al proceso dado.

Si eres usuario experto, debes referirte a la sección: Pasos para análisis.

Por el contrario, si eres desarrollador ó quieres saber cómo se generó esta solución, se sugiere leer todo el documento.

### Consideraciones 260903

- Los archivos recibidos son sumamente grandes, por lo que se intenta hacer el procesamiento por tramos
- El archivo recibido está en el directorio que antecede a este, de nombre ebd_MX_relJun-2025.txt
- Hoy, en los primeros días de septiembre, empezamos las pruebas con un archivo de 2025 con 9Gb de tamaño.
- Un factor fundamental para la carga de información es la cantidad de memoria RAM
- Nuestra maquina de desarrollo tiene RAM de 32Gb.
- Considerando el tamaño del archivo y la RAM, es recomendable utilizar Polars en vez de pandas para la carga en un solo intento.
- Si hacemos el intento con Pandas, sería en segmentos.
- Se opta por la opción de segmentos con Pandas debido a que Polars se trabajaría en Rust, lo que implica una mayor ruta de aprendizaje.
- Vemos que es un archivo txt, por lo cual abrimos las primeras lineas para verificar la distribución de la información y revisar la viabilidad el proceso

### Revisamos las primeras lineas del archivo

Desde una terminal Linux:

```bash
head ebd_MX_relJun-2025.txt
```

Verificamos que es un tipo CSV, separados los campos por tabulares y los registros por saltos de línea.

## Intento de carga por segmentos 260903-1

### Notas de la prueba de todos los registros
- La prueba genera un csv de resultados
- La prueba marca al iniciar la hora y al terminar.
- Se hicieron correcciones en los dtypes para no tener errores de tipo de datos. Todos ellos se hicieron de tipo string.
- Los segmentos son de 50,000 registros.
- La carga toma aproximadamente 9:35 mins

A continuación, definimos el proceso dividido en métodos para su manipulación

## Pasos para el análisis ##

### Escenarios de trabajo
Para la ejecución del análisis primero debemos determinar en qué escenario actuar, de ello depede la rápidez de la ejecución de las pruebas. Lo anterior, porque este análisis se
realiza segmento por segmento, debido a que el tamaño de la importación y procesamiento es muy grande, y lo único que asegura el análisis sin importar el tamaño de la carga 
es haciéndo este tipo de análisis, que no satura la memoria (evitando un bloqueo de sistemas cliente).<br>
A continuación se detalla cada tipo de escenario de acción, así mismo, como configurar cada uno. De ello depende la velocidad de procesamiento:

- **Datos completos.** Usando el txt completo.

  ¿Cómo configurar una prueba con datos completos?

```python
proc_first_chunk = False
proc_reduced_origin = False
```

- **Primer segmento.** El procesamiento se hace por segmentos y utilizando el primer segmento, aseguramos que el procesamiento se realice rápido. Este sería el escenario de trabajo sugerido. La cantidad de registros de este primer segmento depende del valor de la variable *chunk_size*.

  ¿Cómo configurar una prueba con los datos del primer segmento?

```python
proc_first_chunk = True
proc_reduced_origin = False
```

- **Datos de varios segmentos iniciales.** Hasta este momento, este escenario sólo se tiene considerado para propósitos de pruebas de desarrollo. Sirve para asegurar que lo aplicado en el primer segmento, se puede aplicar a los demás, reduciéndo la probabilidad de falla y el tiempo de un procesamiento con datos completos.

  ¿Cómo configurar una prueba con los datos del primer segmento?

```python
proc_first_chunk = False
proc_reduced_origin = True
```

### Variables utlizadas
Se recomienda sólo útilizar las variables explicadas en este documento

|     Variable          |   Tipo   |      Descripción                                      |
| --------------------- | -------- | ----------------------------------------------------- |
| *chunk_size*          |   Num    | tamaño del segmento                                   |
|  *bytes_written*      |   Bol    | No mover - indicador de que se escribió la salida     |
| *src_file*            |   Str    | archivo origen, versión completa                      |
| *src_file_reduced*    |   Str    | archivo origen, versión reducida                      |
| *src_sep*             |   Str    | separador campos archivo origen ("\t" = tabulador)    |
| *result_sep*          |   Str    | separador campos archivo resultado ("\t" = tabulador) |
| *result_file*         |   Str    | archivo de resultado                                  |
| *proc_first_chunk*    |   Bol    | el primer tramo (chuck_size) ó se procesa todo        |
| *proc_reduced_origin* |   Bol    | el primer tramo (chuck_size) ó se procesa todo        |

Posterior a las variables, se detallan los campos que se importan como string, debido a problemas de tipo de dato.

#### Nota de desarrollo -> Generación de un archivo origen reducido.

Para generar el escenario de los *Datos de varios segmentos* se estima el total de registros, basado en el total de líneas del archivo recibido.

*wc -l ebd_MX_relJun-2025.txt*

Para reducir el archivo, ejecutamos el comando:

*head -n -N file.csv > new_file.csv*

En dónde N, es el tamaño de líneas que deseamos eliminar del archivo.

*head -n -23839000 ebd_MX_relJun-2025.txt > ebd_MX_relJun-2025_first.txt*

### PASO 1. Definición de variables para el procesamiento

In [186]:
import sys
from datetime import datetime

import pandas as pda

chunk_size = 100  # tamaño de la muestra
bytes_written = False  # indicador de que se escribió la salida
src_file = "./../ebd_MX_relJun-2025.txt"  # archivo origen, versión completa
src_file_reduced = (
    "./../ebd_MX_relJun-2025_first.txt"  # archivo origen, versión reducida
)
src_sep = "\t"  # separador de campos en archivo origen
result_sep = "\t"  # separador de campos en archivo de resultado
result_file = "./../processed_output.csv"  # archivo de resultado
write_result_file = True
proc_first_chunk = False  # solo el primer tramo (chuck_size) ó se procesa todo
proc_reduced_origin = True  # procesar con archivo reducido

# Las columnas con problemas de lectura se consideran cadenas(str), evitando errores
ebird_dtypes = {
    "BREEDING CODE": str,
    "BREEDING CATEGORY": str,
    "BEHAVIOR CODE": str,
    "AGE/SEX": str,
    "SPECIES COMMENTS": str,
    "OBSERVER ORCID ID": str,
    "PROJECT NAMES": str,
    "GROUP IDENTIFIER": str,
    "OBSERVATION COUNT": str,
    "CHECKLIST COMMENTS": str,
    "SUBSPECIES COMMON NAME": str,
    "SUBSPECIES SCIENTIFIC NAME": str,
    "EXOTIC CODE": str,
    "IBA CODE": str,
    "USFWS CODE": str,
    "PROJECT IDENTIFIERS": str,
}

### Métodos para el procesamiento

#### Nota de desarrollo -> Métodos informativos de procesamiento

In [5]:
def show_start_time():
    start_time = datetime.now()
    print(f"--- Process Started at: {start_time.strftime('%Y-%m-%d %H:%M:%S')} ---")


def show_end_time():
    end_time = datetime.now()
    print(f"--- Process Finished at: {end_time.strftime('%Y-%m-%d %H:%M:%S')} ---")
    duration = end_time - start_time
    print(f"Total time elapsed: {duration}")

#### Nota de desarrollo -> Métodos de carga y escritura de resultados

In [184]:
def write_results_to_csv(chunk, result_file, result_sep, bytes_written):
    chunk.to_csv(
        result_file,
        mode="a",
        index=False,
        header=not bytes_written,
        sep=result_sep,
    )

    return True


def load_chunks(
    src_file,
    ebird_dtypes,
    src_sep,
    chunk_size,
    write_result_file,
    result_file,
    result_sep,
    bytes_written,
):
    start_auto_increment = 1
    for chunk in pd.read_csv(
        src_file,
        sep=src_sep,
        chunksize=chunk_size,
        dtype=ebird_dtypes,
    ):
        # print("BEFORE")
        # print(f"start_auto_increment:{start_auto_increment}")
        start_auto_increment = process(chunk, start_auto_increment, chunk_size)
        # print("AFTER")
        # print(f"start_auto_increment:{start_auto_increment}")
        # print(f"limit:{(start_auto_increment + chunk_size)}")
        # print([*range(start_auto_increment, chunk_size + 1)])
        # sys.exit(0)
        if write_result_file == True:
            bytes_written = write_results_to_csv(
                chunk, result_file, result_sep, bytes_written
            )

    return


def load_first_chunk(
    src_file,
    ebird_dtypes,
    src_sep,
    chunk_size,
    write_result_file,
    result_file,
    result_sep,
    bytes_written,
):
    start_auto_increment = 1
    reader = pd.read_csv(
        src_file,
        sep=src_sep,
        chunksize=chunk_size,
        dtype=ebird_dtypes,
    )
    first_chunk = reader.get_chunk()
    process(first_chunk, start_auto_increment, chunk_size)
    if write_result_file == True:
        bytes_written = write_results_to_csv(
            first_chunk, result_file, result_sep, bytes_written
        )

    return

### PASO 2. Métodos del negocio

Métodos que definen el procesamiento

#### Primer paso de procesamiento - Agregar columnas filtro

In [154]:
def add_filter_cols(chunk, start_auto_increment, chunk_size):
    # print(f"len:{len(chunk)}")
    # print(f"limit:{(start_auto_increment + chunk_size)}")
    # print([*range(start_auto_increment, chunk_size + 1)])
    # sys.exit(0)
    rng_limit = start_auto_increment + len(chunk)
    chunk["id"] = range(start_auto_increment, rng_limit)
    chunk["val_sp"] = 1
    chunk["MRD"] = 1
    chunk["id_cord"] = None
    chunk["gui_snib"] = 1

    return rng_limit

#### Método que va llamando cada procesamiento
Este método va llamando, uno a uno, los anteriores pasos de procesamiento.

In [179]:
def process(chunk, start_auto_increment, chunk_size):
    start_auto_increment = add_filter_cols(chunk, start_auto_increment, chunk_size)

    return start_auto_increment

### PASO 3. Método de ejecución
Echar a andar cada vez que queramos ver el resultado.

In [188]:
show_start_time()

if proc_reduced_origin == True:
    src_file = src_file_reduced

if proc_first_chunk == True:
    load_first_chunk(
        src_file,
        ebird_dtypes,
        src_sep,
        chunk_size,
        write_result_file,
        result_file,
        result_sep,
        bytes_written,
    )
else:
    load_chunks(
        src_file,
        ebird_dtypes,
        src_sep,
        chunk_size,
        write_result_file,
        result_file,
        result_sep,
        bytes_written,
    )
show_end_time()

--- Process Started at: 2026-09-07 16:14:53 ---
--- Process Finished at: 2026-09-07 16:14:54 ---
Total time elapsed: 3 days, 7:09:57.009931
